# Ejercicio 3 — Validación y calidad de datos

**Seminario de Lenguajes Opción Python — TI 2026**

En esta sección se aplican las funciones de validación del módulo `validacion.py`  
sobre los datasets descargados en `raw_datasets/`.

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import validacion

## Configuración

Ajustar las rutas y parámetros según el dataset a analizar.

In [ ]:
# ── Ajustar según el dataset a analizar ──────────────────────────────────────
RUTA_DATASET   = '../raw_datasets/occurrence.txt'   # ruta al archivo principal
SEPARADOR      = '\t'                               # \t para TSV, ',' para CSV
ENCODING       = 'utf-8'                            # encoding del archivo
# ─────────────────────────────────────────────────────────────────────────────

---
## 3.A — Validación de coordenadas geográficas

Detecta registros donde:
- `decimalLatitude` no esté en `[-90, 90]`
- `decimalLongitude` no esté en `[-180, 180]`
- alguno de los valores no pueda convertirse a número

In [ ]:
resultado_3a = validacion.validar_coordenadas(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Registros con coordenadas inválidas: {resultado_3a['cantidad_invalidos']}")
print()

for reg in resultado_3a['registros_invalidos'][:5]:   # mostrar hasta 5
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | Motivos: {reg['_motivos']}")

---
## 3.B — Coordenadas incompletas

Detecta registros donde solo una de las dos coordenadas está presente.

In [ ]:
resultado_3b = validacion.validar_coordenadas_incompletas(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Registros con coordenadas incompletas: {resultado_3b['cantidad']}")

for reg in resultado_3b['registros'][:5]:
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | {reg['_motivo']}")

---
## 3.C — Validación de fechas

Detecta fechas con formato inválido, no interpretables o posteriores al año actual.
Se aplica a todos los atributos cuyo nombre contenga `date`.

In [ ]:
resultado_3c = validacion.validar_fechas(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Registros con fechas inválidas: {resultado_3c['cantidad_invalidos']}")

for reg in resultado_3c['registros_invalidos'][:5]:
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | Motivos: {reg['_motivos']}")

---
## 3.D — Registros duplicados

Detecta IDs que aparecen más de una vez.

In [ ]:
resultado_3d = validacion.detectar_duplicados(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Cantidad de registros duplicados: {resultado_3d['cantidad_duplicados']}")
print(f"IDs repetidos (primeros 10):")

for id_val, conteo in list(resultado_3d['ids_repetidos'].items())[:10]:
    print(f"  {id_val}: aparece {conteo} veces")

---
## 3.E — Validación de countryCode

El campo `countryCode` debe contener códigos ISO 3166-1 alpha-2 válidos.

In [ ]:
resultado_3e = validacion.validar_country_code(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Registros con countryCode inválido: {resultado_3e['cantidad_invalidos']}")

for reg in resultado_3e['registros_invalidos'][:5]:
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | {reg['_motivo']}")

---
## 3.F — Validación de coordinateUncertaintyInMeters

In [ ]:
resultado_3f = validacion.validar_incertidumbre_coordenadas(
    RUTA_DATASET, umbral_maximo=100, separador=SEPARADOR, encoding=ENCODING
)

print(f"Registros con incertidumbre inválida: {resultado_3f['cantidad_invalidos']}")

for reg in resultado_3f['registros_invalidos'][:5]:
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | Motivos: {reg['_motivos']}")

---
## 3.G — Resumen de calidad del dataset

In [ ]:
resumen, reporte = validacion.resumen_calidad(RUTA_DATASET, SEPARADOR, ENCODING)

print(reporte)
print("Diccionario resumen:")
for clave, valor in resumen.items():
    print(f"  {clave}: {valor}")

---
## 3.H — Validación por cotas de América del Sur

Constantes utilizadas:
```
LATITUD_MIN_AS  = -56.0
LATITUD_MAX_AS  =  13.0
LONGITUD_MIN_AS = -82.0
LONGITUD_MAX_AS = -34.0
```

In [ ]:
print("Cotas definidas para América del Sur:")
print(f"  Latitud : [{validacion.LATITUD_MIN_AS}, {validacion.LATITUD_MAX_AS}]")
print(f"  Longitud: [{validacion.LONGITUD_MIN_AS}, {validacion.LONGITUD_MAX_AS}]")
print()

resultado_3h = validacion.validar_coordenadas_america_sur(RUTA_DATASET, SEPARADOR, ENCODING)

print(f"Registros fuera de América del Sur: {resultado_3h['cantidad_fuera']}")

for reg in resultado_3h['registros_fuera'][:5]:
    id_val = validacion._obtener_valor(reg, 'occurrenceID', 'id', 'ID')
    print(f"  ID: {id_val} | Motivos: {reg['_motivos']}")

---
## 3.I — Validaciones agrupadas por campo

In [ ]:
print("=== Validación de LATITUD ===")
resultado_lat = validacion.validar_latitud(RUTA_DATASET, SEPARADOR, ENCODING)
for clave, valor in resultado_lat.items():
    if not isinstance(valor, list):
        print(f"  {clave}: {valor}")

print()
print("=== Validación de LONGITUD ===")
resultado_lon = validacion.validar_longitud(RUTA_DATASET, SEPARADOR, ENCODING)
for clave, valor in resultado_lon.items():
    if not isinstance(valor, list):
        print(f"  {clave}: {valor}")